In [0]:
df_bronze = spark.table("workspace.default.transactions_data")

In [0]:
# Preview a small sample of the raw transactional data
# Used as a basic sanity check to understand column structure and values
display(df_bronze.limit(10))

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
23099344,2019-06-13T18:32:00.000Z,997,2261,$53.14,Chip Transaction,83070,Chula Vista,CA,91913.0,4121,null
23099345,2019-06-13T18:33:00.000Z,89,5584,$19.01,Chip Transaction,23801,Mesilla Park,NM,88047.0,5813,null
23099346,2019-06-13T18:33:00.000Z,1219,3466,$5.36,Chip Transaction,45689,Oxford,MD,21654.0,5651,null
23099348,2019-06-13T18:34:00.000Z,1135,3081,$21.92,Online Transaction,16798,ONLINE,null,null,4121,null
23099349,2019-06-13T18:34:00.000Z,1407,5966,$82.85,Online Transaction,9932,ONLINE,null,null,5311,null
23099350,2019-06-13T18:34:00.000Z,1459,3383,$13.62,Swipe Transaction,87785,Albuquerque,NM,87121.0,5912,null
23099351,2019-06-13T18:34:00.000Z,1776,4938,$57.74,Chip Transaction,63694,Nashport,OH,43830.0,5411,null
23099353,2019-06-13T18:35:00.000Z,200,2875,$13.61,Chip Transaction,20561,Scott Depot,WV,25560.0,5912,null
23099355,2019-06-13T18:37:00.000Z,285,2927,$100.00,Swipe Transaction,27092,San Jose,CA,95111.0,4829,null
23099357,2019-06-13T18:37:00.000Z,1240,1295,$25.18,Chip Transaction,44919,Coshocton,OH,43812.0,5814,null


In [0]:
# Inspect the inferred schema of the Bronze table
# This helps identify incorrect data types and nullable fields
df_bronze.printSchema()

root
 |-- id: long (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: long (nullable = true)
 |-- card_id: long (nullable = true)
 |-- amount: string (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: long (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- mcc: long (nullable = true)
 |-- errors: string (nullable = true)



In [0]:
# Count total number of records in the dataset
# Useful for validating ingestion completeness
df_bronze.count()

13305915

In [0]:
# Check uniqueness of the business key (transaction id)
# Differences between total count and distinct count indicate duplicates
df_bronze.select("id").distinct().count()

13305915

In [0]:
from pyspark.sql.functions import col

# Assess null values in critical columns
# Key fields should not contain nulls in Silver layer
display(
    df_bronze.selectExpr(
    "count(*) as total_rows",
    "sum(case when id is null then 1 else 0 end) as null_id",
    "sum(case when date is null then 1 else 0 end) as null_date",
    "sum(case when amount is null then 1 else 0 end) as null_amount"
    )
)

total_rows,null_id,null_date,null_amount
13305915,0,0,0


In [0]:
from pyspark.sql.functions import col, sum as _sum, count, when

display(
    df_bronze
        .groupBy("use_chip")
        .agg(
            count("*").alias("total_rows"),
            _sum(when(col("id").isNull(), 1).otherwise(0)).alias("null_id"),
            _sum(when(col("date").isNull(), 1).otherwise(0)).alias("null_date"),
            _sum(when(col("amount").isNull(), 1).otherwise(0)).alias("null_amount")
        )
)


use_chip,total_rows,null_id,null_date,null_amount
Online Transaction,1557912,0,0,0
Chip Transaction,4780818,0,0,0
Swipe Transaction,6967185,0,0,0


In [0]:

# Explore distinct values of the amount field
# Amount is currently stored as string and contains currency symbols
display(df_bronze.select("amount").distinct().limit(20))

amount
$11.28
$22.79
$17.04
$113.09
$30.13
$59.25
$31.81
$143.70
$19.04
$28.64


In [0]:
# Attempt a numeric comparison on the amount column
# Expected to fail due to non-numeric characters (currency symbol)
# This error is an important EDA insight for Silver layer design
df_bronze.filter(col("amount") <= 0).count()

---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-6552634864526984>, line 4
      1 # Attempt a numeric comparison on the amount column
      2 # Expected to fail due to non-numeric characters (currency symbol)
      3 # This error is an important EDA insight for Silver layer design
----> 4 df_bronze.filter(col("amount") <= 0).count()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:317, in DataFrame.count(self)
    314 def count(self) -> int:
    315     table, _ = self.agg(
    316         F._invoke_function("count", F.lit(1))
--> 317     )._to_table()  # type: ignore[operator]
    318     return table[0][0].as_py()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1890, in DataFrame._to_table(self)
   1888 def _to_table(self) -> Tuple["pa.Table", Optional[StructType]]:
   1889     query = self._

In [0]:
# Analyze distribution of transaction types
# Used for later standardization in Silver layer
display(df_bronze.groupBy("use_chip").count())

use_chip,count
Online Transaction,1557912
Swipe Transaction,6967185
Chip Transaction,4780818
